# Digital Image Processing – Assignment 8
## Generation and Analysis of Noise Distributions

**Tasks from the assignment**
1. Uniform noise
2. Gaussian noise
3. Erlang noise
4. Exponential noise
5. Rayleigh noise

This notebook generates each noise distribution from its probability distribution, adds it to an image, displays the noisy result, and plots the noise histogram.

> **Important note about `imnoise()`:** `imnoise()` is a MATLAB function, not a Python function. MATLAB's `imnoise` supports Gaussian, salt-and-pepper, speckle and Poisson noise; it does not provide direct modes for all five distributions in this assignment. The notebook therefore compares each equation-based generator with the corresponding built-in Python/SciPy generator where available. A MATLAB `imnoise()` Gaussian comparison cell is also provided near the end. Do not label the Python comparisons as MATLAB `imnoise()` results.

## 1. Import required libraries

We use NumPy for array operations, Matplotlib for displaying images and histograms, SciPy for distribution-based built-in generators, and scikit-image for a sample grayscale image and an additional Gaussian-noise reference.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import gamma, expon, rayleigh
from skimage import data, img_as_float
from skimage.util import random_noise

np.random.seed(42)
print("Libraries imported successfully.")

## 2. Load an image

The notebook uses the standard grayscale `camera` image from scikit-image, so it runs without needing an external image file. Pixel values are normalized to the range [0, 1]. You can replace this with your own grayscale image if required.

In [ ]:
image = img_as_float(data.camera())

plt.figure(figsize=(6, 5))
plt.imshow(image, cmap="gray", vmin=0, vmax=1)
plt.title("Original Grayscale Image")
plt.axis("off")
plt.show()

print("Image shape:", image.shape)
print("Minimum pixel value:", image.min())
print("Maximum pixel value:", image.max())

## 3. Noise-generation equations

For a random variable \(Z\), the probability density functions (PDFs) used here are:

- **Uniform:** \(p(z)=1/(b-a)\), for \(a\le z\le b\).
- **Gaussian:** \(p(z)=\frac{1}{\sqrt{2\pi\sigma^2}}e^{-(z-\mu)^2/(2\sigma^2)}\).
- **Erlang:** \(p(z)=\frac{\lambda^k z^{k-1}e^{-\lambda z}}{(k-1)!}\), for \(z\ge0\), where integer \(k\ge1\).
- **Exponential:** \(p(z)=\lambda e^{-\lambda z}\), for \(z\ge0\).
- **Rayleigh:** \(p(z)=\frac{z}{\sigma^2}e^{-z^2/(2\sigma^2)}\), for \(z\ge0\).

The parameters below are chosen to make the noise visible while keeping the noisy image interpretable. Additive noise is clipped to [0, 1] for display.

## 4. Define equation-based generators and image helper

In [ ]:
def uniform_noise(shape, a=-0.08, b=0.08, rng=None):
    """Generate samples from Uniform(a, b)."""
    rng = np.random.default_rng() if rng is None else rng
    return a + (b - a) * rng.random(shape)


def gaussian_noise(shape, mu=0.0, sigma=0.04, rng=None):
    """Generate Gaussian samples using Z = mu + sigma*N(0,1)."""
    rng = np.random.default_rng() if rng is None else rng
    return mu + sigma * rng.standard_normal(shape)


def erlang_noise(shape, k=3, rate=35.0, rng=None):
    """Erlang(k, rate): sum of k independent exponential(rate) samples."""
    rng = np.random.default_rng() if rng is None else rng
    samples = rng.exponential(scale=1.0 / rate, size=(k,) + tuple(shape))
    return samples.sum(axis=0)


def exponential_noise(shape, rate=18.0, rng=None):
    """Exponential(rate), with PDF rate*exp(-rate*z), z >= 0."""
    rng = np.random.default_rng() if rng is None else rng
    return rng.exponential(scale=1.0 / rate, size=shape)


def rayleigh_noise(shape, sigma=0.035, rng=None):
    """Rayleigh(sigma), generated by inverse transform sampling."""
    rng = np.random.default_rng() if rng is None else rng
    u = np.clip(rng.random(shape), 1e-12, 1 - 1e-12)
    return sigma * np.sqrt(-2.0 * np.log(1.0 - u))


def add_noise_to_image(img, noise):
    """Add noise and clip output to the valid image range [0, 1]."""
    return np.clip(img + noise, 0.0, 1.0)


print("Noise-generation functions defined.")

## 5. Generate all five noise types

Each noise array is generated using the equations above. We retain the noise arrays separately so that their histograms show the distribution itself, rather than the histogram of the noisy image.

In [ ]:
rng = np.random.default_rng(42)

noise_arrays = {
    "Uniform": uniform_noise(image.shape, a=-0.08, b=0.08, rng=rng),
    "Gaussian": gaussian_noise(image.shape, mu=0.0, sigma=0.04, rng=rng),
    "Erlang": erlang_noise(image.shape, k=3, rate=35.0, rng=rng),
    "Exponential": exponential_noise(image.shape, rate=18.0, rng=rng),
    "Rayleigh": rayleigh_noise(image.shape, sigma=0.035, rng=rng),
}

noisy_images = {
    name: add_noise_to_image(image, noise)
    for name, noise in noise_arrays.items()
}

print("Generated noise arrays:", list(noise_arrays.keys()))

## 6. Display the five noisy images

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 9))
axes = axes.ravel()

axes[0].imshow(image, cmap="gray", vmin=0, vmax=1)
axes[0].set_title("Original")

for ax, (name, noisy) in zip(axes[1:], noisy_images.items()):
    ax.imshow(noisy, cmap="gray", vmin=0, vmax=1)
    ax.set_title(name + " Noise")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

## 7. Plot histograms of the generated noise

The histogram represents how frequently different noise values occur. The expected shapes are:
- Uniform: approximately flat between its lower and upper limits.
- Gaussian: bell-shaped and symmetric around its mean.
- Erlang: nonnegative, typically right-skewed for the selected shape parameter.
- Exponential: highest near zero and decreasing to the right.
- Rayleigh: starts at zero, rises to a mode, then decreases with a right tail.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 9))
axes = axes.ravel()

for ax, (name, noise) in zip(axes, noise_arrays.items()):
    ax.hist(noise.ravel(), bins=80, density=True, alpha=0.8)
    ax.set_title(name + " Noise Histogram")
    ax.set_xlabel("Noise value")
    ax.set_ylabel("Probability density")
    ax.grid(alpha=0.2)

axes[-1].axis("off")
plt.tight_layout()
plt.show()

## 8. Compare equation-based generators with built-in Python/SciPy generators

The following cell generates a second sample using established library functions for the same distributions and plots the two histograms together. The comparison checks whether the equation-based generator and library generator produce similar distribution shapes. Random samples will not be identical because they are independently generated.

In [ ]:
rng = np.random.default_rng(123)
built_in_noise = {
    "Uniform": rng.uniform(-0.08, 0.08, size=image.shape),
    "Gaussian": rng.normal(0.0, 0.04, size=image.shape),
    "Erlang": gamma.rvs(a=3, scale=1/35.0, size=image.shape, random_state=rng),
    "Exponential": expon.rvs(scale=1/18.0, size=image.shape, random_state=rng),
    "Rayleigh": rayleigh.rvs(scale=0.035, size=image.shape, random_state=rng),
}

fig, axes = plt.subplots(2, 3, figsize=(14, 9))
axes = axes.ravel()

for ax, name in zip(axes, noise_arrays.keys()):
    ax.hist(noise_arrays[name].ravel(), bins=80, density=True,
            histtype="step", linewidth=1.5, label="Equation-based")
    ax.hist(built_in_noise[name].ravel(), bins=80, density=True,
            histtype="step", linewidth=1.5, label="Built-in library")
    ax.set_title(name + " Distribution Comparison")
    ax.set_xlabel("Noise value")
    ax.set_ylabel("Probability density")
    ax.legend()
    ax.grid(alpha=0.2)

axes[-1].axis("off")
plt.tight_layout()
plt.show()

## 9. Compare the noisy images from both generators

In [ ]:
fig, axes = plt.subplots(5, 2, figsize=(10, 16))

for row, name in enumerate(noise_arrays.keys()):
    axes[row, 0].imshow(add_noise_to_image(image, noise_arrays[name]),
                        cmap="gray", vmin=0, vmax=1)
    axes[row, 0].set_title(name + " – Equation-based")
    axes[row, 1].imshow(add_noise_to_image(image, built_in_noise[name]),
                        cmap="gray", vmin=0, vmax=1)
    axes[row, 1].set_title(name + " – Built-in Python/SciPy")

    axes[row, 0].axis("off")
    axes[row, 1].axis("off")

plt.tight_layout()
plt.show()

## 10. Numerical comparison of sample statistics

Theoretical mean and variance are useful checks:
- Uniform(a,b): mean \((a+b)/2\), variance \((b-a)^2/12\).
- Gaussian(\(\mu,\sigma\)): mean \(\mu\), variance \(\sigma^2\).
- Erlang(k, rate): mean \(k/\lambda\), variance \(k/\lambda^2\).
- Exponential(rate): mean \(1/\lambda\), variance \(1/\lambda^2\).
- Rayleigh(\(\sigma\)): mean \(\sigma\sqrt{\pi/2}\), variance \((4-\pi)\sigma^2/2\).

In [ ]:
parameters = {
    "Uniform": (0.0, (0.16**2) / 12),
    "Gaussian": (0.0, 0.04**2),
    "Erlang": (3/35.0, 3/(35.0**2)),
    "Exponential": (1/18.0, 1/(18.0**2)),
    "Rayleigh": (0.035*np.sqrt(np.pi/2),
                 ((4-np.pi)/2)*(0.035**2)),
}

print(f"{'Distribution':<14} {'Sample mean':>14} {'Theory mean':>14} "
      f"{'Sample var':>14} {'Theory var':>14}")
print("-" * 76)

for name, noise in noise_arrays.items():
    sample_mean = np.mean(noise)
    sample_var = np.var(noise)
    theory_mean, theory_var = parameters[name]
    print(f"{name:<14} {sample_mean:14.6f} {theory_mean:14.6f} "
          f"{sample_var:14.6f} {theory_var:14.6f}")

## 11. MATLAB `imnoise()` comparison (Gaussian)

This is the MATLAB command for a Gaussian-noise comparison. Run this section in MATLAB, not in the Python notebook. MATLAB's `imnoise` uses a Gaussian mean and variance, whereas the Python Gaussian generator above uses standard deviation `sigma`; therefore set MATLAB variance to `sigma**2`.

```matlab
I = im2double(imread('cameraman.tif'));
mu = 0;
sigma = 0.04;

% MATLAB built-in Gaussian noise
J_imnoise = imnoise(I, 'gaussian', mu, sigma^2);

% Display comparison
figure;
subplot(1,2,1); imshow(I); title('Original Image');
subplot(1,2,2); imshow(J_imnoise); title('Gaussian Noise using imnoise');

% Histogram of the added noise (before clipping)
% imnoise output is clipped, so this is an approximate noise histogram.
noise_imnoise = J_imnoise - I;
figure;
histogram(noise_imnoise(:), 80, 'Normalization', 'pdf');
title('Histogram of noise produced by imnoise');
xlabel('Noise value'); ylabel('Probability density');
```

**Important:** `imnoise(I,'gaussian',mu,var)` adds Gaussian noise to an image. It does not generate Uniform, Erlang, Exponential or Rayleigh noise. Those distributions must be generated separately (for example, using the equations in this notebook) if you need all five assignment outputs.

## 12. Conclusion

In this assignment, five noise distributions—Uniform, Gaussian, Erlang, Exponential and Rayleigh—were generated using their probability models and added to a grayscale image. Histograms were plotted to observe the characteristic shape of each distribution. The equation-based samples were compared with built-in NumPy/SciPy distribution generators, and a MATLAB `imnoise()` Gaussian comparison was provided separately.

The histogram shapes should broadly match the expected PDFs. Small differences between the two sets of samples are normal because random samples differ from run to run.